# 📖 AI Document & Book Scanner Engine
### Next-Gen 4-Corner Quad Rectification, Finger Removal, Paper Whitening & PDF Export

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

> **Environment Note:** Ensure your Colab runtime is set to GPU: **Runtime ➔ Change runtime type ➔ T4 GPU**.

In [ ]:
# @title 🔄 Git Sync: Pull Latest Updates
# Run this cell anytime new updates or improvements are pushed to GitHub!
!git pull 2>/dev/null || echo "Working directory is ready."


In [ ]:
# @title 🚀 Step 1: Environment Setup & Dependency Installation
import os
import sys
import subprocess

print("Checking GPU acceleration...")
try:
    import torch
    print(f"CUDA Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("⚠️ Running on CPU. For maximum processing speed, switch to GPU (Runtime -> Change runtime type -> T4 GPU)")
except ImportError:
    print("PyTorch not detected. Installing dependencies...")

print("\nInstalling core vision, inpainting, and PDF compilation libraries...")
!apt-get update -qq > /dev/null
!apt-get install -y -qq tesseract-ocr libtesseract-dev > /dev/null
!pip install -q opencv-python-headless img2pdf pytesseract mediapipe timm einops gradio

print("\n✅ All dependencies successfully installed!")

## 📥 Step 2: Initialize Test Environment & Sample Photos
Loads available repository benchmark images or user photos from `input_images/` into `test_inputs/`.

In [ ]:
# @title 📂 Step 2: Initialize Input Directories
import os
import shutil
import glob
import cv2
import numpy as np

os.makedirs("test_inputs", exist_ok=True)
os.makedirs("user_test_images", exist_ok=True)
os.makedirs("pipeline_outputs", exist_ok=True)

# If repository input_images exists, copy them into test_inputs
repo_samples = glob.glob("input_images/*.jpg") + glob.glob("input_images/*.png")
if repo_samples:
    for s in repo_samples:
        dest = os.path.join("test_inputs", os.path.basename(s))
        if not os.path.exists(dest):
            shutil.copy(s, dest)
    print(f"Loaded {len(repo_samples)} sample images from repository into ./test_inputs/")
else:
    print("Ready for input images in ./test_inputs/ or ./user_test_images/")


## ⚙️ Step 3: Core Pipeline Engines
1. **Pre-Processing:** Auto-Orientation ($0^\circ, 90^\circ, 180^\circ, 270^\circ$) & Deskew
2. **4-Corner Quad Rectification:** Perspective Homography to isolate page from background
3. **Adaptive Dewarping:** Optional gentle curvature compensation
4. **Finger/Thumb Removal:** Margin skin detection & inpainting
5. **Illumination Regularization:** vFlat/CamScanner-style paper whitening & ink contrast boost

In [ ]:
# @title 📐 Pipeline Modules: Orientation, 4-Corner Quad Warp & Whitening
import cv2
import numpy as np
import pytesseract
import time

def order_points(pts: np.ndarray) -> np.ndarray:
    """Orders 4 points as: top-left, top-right, bottom-right, bottom-left"""
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect

class DocumentPreprocessingEngine:
    @staticmethod
    def detect_and_fix_orientation(image: np.ndarray) -> np.ndarray:
        try:
            small = cv2.resize(image, (640, int(640 * image.shape[0] / image.shape[1])))
            osd = pytesseract.image_to_osd(small, output_type=pytesseract.Output.DICT)
            angle = osd.get('rotate', 0)
            if angle == 90:
                return cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)
            elif angle == 180:
                return cv2.rotate(image, cv2.ROTATE_180)
            elif angle == 270:
                return cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE)
        except Exception:
            pass
        return image

    @staticmethod
    def deskew(image: np.ndarray, max_angle: float = 15.0) -> np.ndarray:
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        edges = cv2.Canny(gray, 50, 150, apertureSize=3)
        lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=100, minLineLength=100, maxLineGap=10)
        if lines is None:
            return image
        
        lines = lines.reshape(-1, 4)
        angles = []
        for x1, y1, x2, y2 in lines:
            theta = np.degrees(np.arctan2(float(y2 - y1), float(x2 - x1)))
            if abs(theta) <= max_angle:
                angles.append(theta)
            elif abs(abs(theta) - 90) <= max_angle:
                angles.append(theta - 90 if theta > 0 else theta + 90)
        
        if not angles:
            return image
        
        median_angle = float(np.median(angles))
        if abs(median_angle) < 0.2:
            return image
            
        h, w = image.shape[:2]
        center = (w // 2, h // 2)
        rot_mat = cv2.getRotationMatrix2D(center, median_angle, 1.0)
        return cv2.warpAffine(image, rot_mat, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

class DocumentGeometryEngine:
    @staticmethod
    def detect_and_warp_quad(image: np.ndarray) -> np.ndarray:
        """
        Detects 4 outer page corners using bilateral contour analysis.
        Warps perspective to an un-slanted, flat rectangular page.
        """
        orig = image.copy()
        h, w = image.shape[:2]
        
        scale = 1000.0 / max(h, w)
        small_h, small_w = int(h * scale), int(w * scale)
        small = cv2.resize(image, (small_w, small_h), interpolation=cv2.INTER_AREA)
        
        gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
        blurred = cv2.bilateralFilter(gray, 9, 75, 75)
        edges = cv2.Canny(blurred, 30, 120)
        
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
        closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel, iterations=3)
        
        contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        best_quad = None
        max_area = 0
        total_area = small_w * small_h
        
        sorted_contours = sorted(contours, key=cv2.contourArea, reverse=True)[:5]
        for c in sorted_contours:
            area = cv2.contourArea(c)
            if area < total_area * 0.40:
                continue
            peri = cv2.arcLength(c, True)
            approx = cv2.approxPolyDP(c, 0.02 * peri, True)
            if len(approx) == 4 and area > max_area:
                best_quad = approx.reshape(4, 2)
                max_area = area
                break
                
        if best_quad is None and sorted_contours:
            largest = sorted_contours[0]
            if cv2.contourArea(largest) > total_area * 0.40:
                hull = cv2.convexHull(largest)
                peri = cv2.arcLength(hull, True)
                approx = cv2.approxPolyDP(hull, 0.03 * peri, True)
                if len(approx) == 4:
                    best_quad = approx.reshape(4, 2)
                    
        if best_quad is not None:
            pts = best_quad / scale
            rect = order_points(pts)
            (tl, tr, br, bl) = rect
            
            width_a = np.linalg.norm(br - bl)
            width_b = np.linalg.norm(tr - tl)
            max_w = max(int(width_a), int(width_b))
            
            height_a = np.linalg.norm(tr - br)
            height_b = np.linalg.norm(tl - bl)
            max_h = max(int(height_a), int(height_b))
            
            dst = np.array([
                [0, 0],
                [max_w - 1, 0],
                [max_w - 1, max_h - 1],
                [0, max_h - 1]
            ], dtype="float32")
            
            M = cv2.getPerspectiveTransform(rect, dst)
            return cv2.warpPerspective(orig, M, (max_w, max_h), flags=cv2.INTER_CUBIC)
            
        return orig

    @staticmethod
    def dewarp_3d_surface(image: np.ndarray, strength: float = 0.0) -> np.ndarray:
        if strength <= 0.0:
            return image
        h, w = image.shape[:2]
        grid_x, grid_y = np.meshgrid(np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32))
        curve_profile = np.sin(np.linspace(0, np.pi, h))[:, None]
        horizontal_decay = np.exp(-((grid_x - (w * 0.15)) / (w * 0.25)) ** 2)
        displacement_x = curve_profile * horizontal_decay * (w * strength)
        map_x = np.clip(grid_x - displacement_x, 0, w - 1).astype(np.float32)
        map_y = grid_y.astype(np.float32)
        return cv2.remap(image, map_x, map_y, interpolation=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

class OcclusionRemovalEngine:
    @staticmethod
    def detect_finger_mask(image: np.ndarray) -> np.ndarray:
        h, w = image.shape[:2]
        mask = np.zeros((h, w), dtype=np.uint8)
        border_mask = np.zeros((h, w), dtype=np.uint8)
        border_w = int(w * 0.10)
        border_h = int(h * 0.10)
        border_mask[:border_h, :] = 255
        border_mask[-border_h:, :] = 255
        border_mask[:, :border_w] = 255
        border_mask[:, -border_w:] = 255
        
        ycrcb = cv2.cvtColor(image, cv2.COLOR_BGR2YCrCb)
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
        skin_ycrcb = cv2.inRange(ycrcb, np.array([0, 133, 77]), np.array([255, 173, 127]))
        skin_hsv = cv2.inRange(hsv, np.array([0, 25, 50]), np.array([30, 220, 255]))
        combined_skin = cv2.bitwise_and(skin_ycrcb, skin_hsv)
        candidate_mask = cv2.bitwise_and(combined_skin, combined_skin, mask=border_mask)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
        candidate_mask = cv2.morphologyEx(candidate_mask, cv2.MORPH_CLOSE, kernel, iterations=2)
        contours, _ = cv2.findContours(candidate_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for c in contours:
            if cv2.contourArea(c) > (w * h * 0.002):
                cv2.drawContours(mask, [c], -1, 255, -1)
        return mask

    @staticmethod
    def inpaint_fingers(image: np.ndarray, mask: np.ndarray) -> np.ndarray:
        if np.count_nonzero(mask) == 0:
            return image
        dilated_mask = cv2.dilate(mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7)), iterations=2)
        return cv2.inpaint(image, dilated_mask, inpaintRadius=5, flags=cv2.INPAINT_TELEA)

class IlluminationRegularizationEngine:
    @staticmethod
    def whiten_paper_vflat_style(image: np.ndarray, whiteness_gain: float = 1.15) -> np.ndarray:
        """
        1. Estimates smooth background illumination in LAB luminance space.
        2. Neutralizes shadows and maps paper to clean white.
        3. Preserves blue ink, red margin lines, and pen strokes with high contrast.
        """
        lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        h, w = l.shape
        scale = 800.0 / max(h, w)
        sw, sh = int(w * scale), int(h * scale)
        l_small = cv2.resize(l, (sw, sh), interpolation=cv2.INTER_AREA)
        
        k_size = 51
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (k_size, k_size))
        bg_small = cv2.morphologyEx(l_small, cv2.MORPH_CLOSE, kernel)
        bg_small = cv2.GaussianBlur(bg_small, (51, 51), 0)
        bg = cv2.resize(bg_small, (w, h), interpolation=cv2.INTER_CUBIC)
        
        l_float = l.astype(np.float32)
        bg_float = np.maximum(bg.astype(np.float32), 1.0)
        l_norm = (l_float / bg_float) * 235.0
        
        l_white = np.clip(l_norm * whiteness_gain, 0, 255)
        paper_mask = l_white > 220
        l_white[paper_mask] = 220 + (l_white[paper_mask] - 220) * 1.0
        l_final = np.clip(l_white, 0, 255).astype(np.uint8)
        
        lab_clean = cv2.merge([l_final, a, b])
        result = cv2.cvtColor(lab_clean, cv2.COLOR_LAB2BGR)
        
        blur = cv2.GaussianBlur(result, (0, 0), 3.0)
        enhanced = cv2.addWeighted(result, 1.2, blur, -0.2, 0)
        return np.clip(enhanced, 0, 255).astype(np.uint8)

print("Engine 1 to 5 loaded successfully.")

## 📄 Step 4: End-to-End Orchestrator & PDF Compiler
Orchestrates all engines and compiles output into a standardized **multi-page PDF**.

In [ ]:
# @title 📦 End-to-End Pipeline & Multi-Page PDF Compilation
import img2pdf
from PIL import Image
import glob

class ScannerPipelineOrchestrator:
    def __init__(self, enable_orientation: bool = True, enable_deskew: bool = True,
                 enable_crop: bool = True, enable_dewarp: bool = False,
                 enable_finger_removal: bool = True, enable_whitening: bool = True):
        self.enable_orientation = enable_orientation
        self.enable_deskew = enable_deskew
        self.enable_crop = enable_crop
        self.enable_dewarp = enable_dewarp
        self.enable_finger_removal = enable_finger_removal
        self.enable_whitening = enable_whitening
        
    def process_frame(self, image: np.ndarray) -> dict:
        timings = {}
        stages = {'0_raw': image.copy()}
        current = image.copy()
        
        # 1. Orientation & Deskew
        t0 = time.time()
        if self.enable_orientation:
            current = DocumentPreprocessingEngine.detect_and_fix_orientation(current)
        if self.enable_deskew:
            current = DocumentPreprocessingEngine.deskew(current)
        timings['orientation_deskew_ms'] = round((time.time() - t0) * 1000, 1)
        stages['1_oriented'] = current.copy()
        
        # 2. 4-Corner Quad Perspective Warp & Isolation
        t0 = time.time()
        if self.enable_crop:
            current = DocumentGeometryEngine.detect_and_warp_quad(current)
        timings['crop_perspective_ms'] = round((time.time() - t0) * 1000, 1)
        stages['2_cropped_quad'] = current.copy()
        
        # 3. 3D Dewarp
        t0 = time.time()
        if self.enable_dewarp:
            current = DocumentGeometryEngine.dewarp_3d_surface(current, strength=0.02)
        timings['dewarp_3d_ms'] = round((time.time() - t0) * 1000, 1)
        stages['3_dewarped'] = current.copy()
        
        # 4. Finger Removal
        t0 = time.time()
        if self.enable_finger_removal:
            mask = OcclusionRemovalEngine.detect_finger_mask(current)
            stages['finger_mask'] = mask.copy()
            current = OcclusionRemovalEngine.inpaint_fingers(current, mask)
        timings['finger_removal_ms'] = round((time.time() - t0) * 1000, 1)
        stages['4_inpainted'] = current.copy()
        
        # 5. Paper Whitening & Illumination Regularization
        t0 = time.time()
        if self.enable_whitening:
            current = IlluminationRegularizationEngine.whiten_paper_vflat_style(current)
        timings['whitening_ms'] = round((time.time() - t0) * 1000, 1)
        stages['5_whitened_final'] = current.copy()
        
        total_ms = sum(timings.values())
        timings['total_pipeline_ms'] = round(total_ms, 1)
        return {'final': current, 'stages': stages, 'timings': timings}

    @staticmethod
    def compile_batch_to_pdf(processed_image_paths: list, output_pdf_path: str = "scanned_document.pdf") -> str:
        if not processed_image_paths:
            raise ValueError("No processed image files provided for PDF compilation.")
            
        a4_in_pt = (img2pdf.mm_to_pt(210), img2pdf.mm_to_pt(297))
        layout_fun = img2pdf.get_layout_fun(pagesize=a4_in_pt, fit=img2pdf.FitMode.into)
        
        with open(output_pdf_path, "wb") as f:
            f.write(img2pdf.convert(processed_image_paths, layout_fun=layout_fun))
            
        print(f"✅ Successfully created Multi-Page PDF: {output_pdf_path} ({len(processed_image_paths)} pages)")
        return output_pdf_path

print("Orchestrator and PDF compilation engine ready!")

## 📤 Step 5: Test With Your Own Images (Batch Mode ➔ Multi-Page PDF)
Processes all images in `user_test_images/` (or lets you upload new ones), and creates `My_Scanned_Book.pdf`.

In [ ]:
# @title 🚀 Process Your Images & Generate PDF
from google.colab import files
import glob
import os
import matplotlib.pyplot as plt

user_upload_dir = "user_test_images"
os.makedirs(user_upload_dir, exist_ok=True)
os.makedirs("pipeline_outputs/user_processed", exist_ok=True)

# Check if images already exist in user_test_images or input_images
image_extensions = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')
input_images = []
for ext in image_extensions:
    input_images.extend(glob.glob(os.path.join(user_upload_dir, ext)))
    input_images.extend(glob.glob(os.path.join("input_images", ext)))
input_images = sorted(list(set(input_images)))

if not input_images:
    print("📁 No images found in ./user_test_images/. Click below to upload:")
    uploaded = files.upload()
    if uploaded:
        for fn in uploaded.keys():
            target = os.path.join(user_upload_dir, fn)
            with open(target, 'wb') as f:
                f.write(uploaded[fn])
            input_images.append(target)

if not input_images:
    print("⚠️ No images provided.")
else:
    print(f"\n🚀 Starting batch processing of {len(input_images)} image(s)...")
    orchestrator = ScannerPipelineOrchestrator(enable_dewarp=False) # planar rectifying enabled
    processed_output_paths = []
    
    for idx, img_path in enumerate(input_images):
        filename = os.path.basename(img_path)
        print(f"\nProcessing [{idx+1}/{len(input_images)}]: {filename}")
        raw_bgr = cv2.imread(img_path)
        if raw_bgr is None:
            print(f"  ❌ Failed to decode {filename}, skipping.")
            continue
            
        res = orchestrator.process_frame(raw_bgr)
        
        out_filename = f"page_{idx+1:03d}_cleaned.jpg"
        out_path = os.path.join("pipeline_outputs/user_processed", out_filename)
        cv2.imwrite(out_path, res['final'])
        processed_output_paths.append(out_path)
        
        # Display Before vs After
        fig, axes = plt.subplots(1, 2, figsize=(14, 7))
        axes[0].imshow(cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB))
        axes[0].set_title(f"Original: {filename}", fontsize=13)
        axes[0].axis('off')
        
        axes[1].imshow(cv2.cvtColor(res['final'], cv2.COLOR_BGR2RGB))
        axes[1].set_title(f"Cleaned ({res['timings']['total_pipeline_ms']} ms)", fontsize=13)
        axes[1].axis('off')
        plt.tight_layout()
        plt.show()
        
    if processed_output_paths:
        final_pdf_path = "pipeline_outputs/My_Scanned_Book.pdf"
        ScannerPipelineOrchestrator.compile_batch_to_pdf(processed_output_paths, final_pdf_path)
        print(f"\n🎉 Successfully generated: {final_pdf_path} ({len(processed_output_paths)} pages)")
        files.download(final_pdf_path)


## 🌐 Step 6: Interactive Web Playground (Gradio)
Launch an interactive visual playground inside Colab.

In [ ]:
# @title 🎛️ Launch Interactive Web App (Gradio)
import gradio as gr

def scan_interface(input_image, enable_orientation, enable_dewarp, enable_finger, enable_whitening):
    if input_image is None:
        return None, None, "Please provide an input image."
        
    bgr = cv2.cvtColor(input_image, cv2.COLOR_RGB2BGR)
    
    orch = ScannerPipelineOrchestrator(
        enable_orientation=enable_orientation,
        enable_deskew=True,
        enable_crop=True,
        enable_dewarp=enable_dewarp,
        enable_finger_removal=enable_finger,
        enable_whitening=enable_whitening
    )
    
    res = orch.process_frame(bgr)
    final_bgr = res['final']
    
    os.makedirs("pipeline_outputs", exist_ok=True)
    page_path = "pipeline_outputs/web_scanned_page.jpg"
    cv2.imwrite(page_path, final_bgr)
    
    pdf_out = "pipeline_outputs/scanned_single.pdf"
    ScannerPipelineOrchestrator.compile_batch_to_pdf([page_path], pdf_out)
    
    final_rgb = cv2.cvtColor(final_bgr, cv2.COLOR_BGR2RGB)
    timing_str = "\n".join([f"{k}: {v} ms" for k, v in res['timings'].items()])
    
    return final_rgb, pdf_out, timing_str

with gr.Blocks(title="AI Book Scanner Engine") as demo:
    gr.Markdown("## 📖 AI Document & Book Scanner Playground")
    gr.Markdown("Upload a photo to apply 4-corner perspective rectification, finger removal, paper whitening, and export as PDF.")
    
    with gr.Row():
        with gr.Column():
            input_img = gr.Image(type="numpy", label="Raw Document Photo")
            with gr.Accordion("Pipeline Configuration", open=True):
                chk_orient = gr.Checkbox(value=True, label="Auto-Orientation (0/90/180/270)")
                chk_crop = gr.Checkbox(value=True, label="4-Corner Quad Perspective Rectification")
                chk_dewarp = gr.Checkbox(value=False, label="3D Spine Surface Dewarping (for heavily curved books)")
                chk_finger = gr.Checkbox(value=True, label="Finger / Thumb Inpainting")
                chk_white = gr.Checkbox(value=True, label="Paper Whitening & Shadow Removal")
            btn_process = gr.Button("🚀 Process Page & Generate PDF", variant="primary")
            
        with gr.Column():
            output_img = gr.Image(type="numpy", label="Processed Scanned Output")
            output_pdf = gr.File(label="Download High-Resolution PDF")
            output_timings = gr.Textbox(label="Execution Timing Breakdown", lines=6)
            
    btn_process.click(
        fn=scan_interface,
        inputs=[input_img, chk_orient, chk_dewarp, chk_finger, chk_white],
        outputs=[output_img, output_pdf, output_timings]
    )

demo.launch(share=True, debug=False)